# 04 – Feature Engineering

Dieses Notebook erstellt domänenspezifische Features für das ML-Modell.

**Feature-Kategorien:**
- Kalendarisch: Wochentag, Wochenende, Heizperiode
- Thermodynamisch: Temperaturdelta, thermische Trägheit
- Preisbezogen: Lag-Features, gleitende Durchschnitte
- Gebäudespezifisch: Renovierungsindex, Gebäudetyp-Dummies
- Solar: Potenzial aus PV-Größe × Sonnenscheindauer

**Selektion:** Korrelations-Check (Schwellenwert 0.90) + SHAP-basierte Importance

In [ ]:
%pip install pandera polars xgboost matplotlib seaborn lightgbm optuna

In [8]:
import sys
import numpy as np
import polars as pl
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor

current_dir = Path('..') / '03_src'
sys.path.append(str(current_dir))
root_dir = Path('..')

from config import target_col, feature_cols, feature_cols_cleaned
from data_cleaner import check_amount_nulls
from features import (
    add_weekdays, add_weekend, heating_season, renovation_index,
    heating_amount, add_price_features, solar_potentials,
    add_thermal_dynamics, add_cyclic_features, add_building_type_dummies,
    add_heatpump, add_pv
)
from utilis import train_test_split, correlated_features_drop, feature_importance_analyse, compare_models, tune_lightgbm

## 4.1 Daten laden

In [2]:
data_combined_cleaned = pl.read_parquet(
    str(root_dir / '02_data' / 'processed' / 'combined_data_cleaned.parquet')
)

## 4.2 Features erstellen

In [3]:
df_ml = (
    data_combined_cleaned
    .pipe(add_weekdays)             # Wochentag (1-7)
    .pipe(add_weekend)              # Wochenend-Dummy
    .pipe(heating_season)           # Heizperiode (Okt-Mär)
    .pipe(renovation_index)         # Summe Renovierungs-Dummies
    .pipe(heating_amount)           # Fläche × Bewohner
    .pipe(add_price_features)       # Lag-Features & gleitende Durchschnitte
    .pipe(solar_potentials)         # PV-Größe × Sonnenscheindauer
    .pipe(add_thermal_dynamics)     # EMA 3d + Temperaturdelta
    .pipe(add_cyclic_features)      # Sinus/Kosinus für Monate
    .pipe(add_building_type_dummies)# One-Hot-Encoding Gebäudetyp
    .pipe(add_heatpump) # Wärmepumpen Info
    .pipe(add_pv)# PV Info
    .select(['date', target_col] + feature_cols)
)

print(f'Features: {len(feature_cols)} | Zielvariable: {target_col}')
check_amount_nulls(df_ml)

Features: 21 | Zielvariable: kwh_received_total

---------------------------------------------
TOP 20 SPALTEN MIT MEISTEN NULLS
Datensatz: 937,456 Zeilen | 23 Spalten
Spalten mit Nulls: 1
---------------------------------------------
shape: (1, 3)
┌─────────────────────────┬────────────┬───────┐
│ spalte                  ┆ null_count ┆ pct   │
│ ---                     ┆ ---        ┆ ---   │
│ str                     ┆ u32        ┆ f64   │
╞═════════════════════════╪════════════╪═══════╡
│ solar_thermal_potential ┆ 101171     ┆ 10.79 │
└─────────────────────────┴────────────┴───────┘


spalte,null_count,pct
str,u32,f64
"""solar_thermal_potential""",101171,10.79


In [5]:
df_ml

date,kwh_received_total,kwh_received_heatpump,kwh_returned_total,temperature_avg_daily,building_type_multi_family_house,building_type_other,building_type_single_family_house,swissix_base,weekday,is_weekend,is_heating_season,renovation_score,heating_amount,price_lag_30d,price_relative_to_month,solar_thermal_potential,temp_inertia_ema_3d,temp_delta_1d,month_sin,month_cos,is_heatpump,is_pv
date,f64,f64,f64,f64,i8,i8,i8,f64,i8,i32,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,i8,i8
2019-03-03,18.33,0.0,8.64,8.4,0,0,0,22.79,7,1,1,0,0.0,22.79,1.022139,0.0,8.4,0.0,1.0,6.1232e-17,0,1
2019-03-04,15.03,0.0,9.04,7.8,0,0,0,37.938333,1,0,1,0,0.0,22.79,1.022139,0.0,8.0,-0.6,1.0,6.1232e-17,0,1
2019-03-05,16.69,0.0,4.57,7.4,0,0,0,37.9225,2,0,1,0,0.0,22.79,1.022139,0.0,7.657143,-0.4,1.0,6.1232e-17,0,1
2019-03-06,29.52,0.0,15.27,8.4,0,0,0,40.97875,3,0,1,0,0.0,22.79,1.022139,0.0,8.053333,1.0,1.0,6.1232e-17,0,1
2019-03-07,16.81,0.0,14.43,9.0,0,0,0,34.819167,4,0,1,0,0.0,22.79,1.022139,0.0,8.541935,0.6,1.0,6.1232e-17,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2024-03-17,10.14,0.0,5.7,7.7,0,0,0,60.337917,7,1,1,0,0.0,65.44125,0.856396,null,7.699999,0.0,1.0,6.1232e-17,0,1
2024-03-18,12.9,0.0,0.65,7.7,0,0,0,81.739583,1,0,1,0,0.0,67.554583,1.152423,null,7.699999,0.0,1.0,6.1232e-17,0,1
2024-03-19,13.83,0.0,0.16,7.7,0,0,0,84.41875,2,0,1,0,0.0,56.08875,1.174558,null,7.7,0.0,1.0,6.1232e-17,0,1


## 4.3 Train-Test-Split & Korrelations-Check

In [9]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: c:\Users\maxkr\.pyenv\pyenv-win\versions\3.11.6\python.exe -m pip install --upgrade pip


In [4]:
# Chronologischer 80/20 Split
X_train, X_test, y_train, y_test = train_test_split(
    df_ml, feature_cols=feature_cols, target_col=target_col
)

# Korrelierte Features entfernen (Schwellenwert 0.90)
X_train_cleaned, X_test_cleaned, _ = correlated_features_drop(
    X_train, X_test, feautre_cols=feature_cols
)
print(f'Features nach Korrelations-Check: {X_train_cleaned.shape[1]}')

Entfernte redundante Features: ['temp_inertia_ema_3d', 'is_heatpump']
Verbleibende Features (19): ['kwh_received_heatpump', 'kwh_returned_total', 'temperature_avg_daily', 'building_type_multi_family_house', 'building_type_other', 'building_type_single_family_house', 'swissix_base', 'weekday', 'is_weekend', 'is_heating_season', 'renovation_score', 'heating_amount', 'price_lag_30d', 'price_relative_to_month', 'solar_thermal_potential', 'temp_delta_1d', 'month_sin', 'month_cos', 'is_pv']
Features nach Korrelations-Check: 19


## 4.4 SHAP Feature-Importance-Analyse

In [5]:
# RandomForest als Basis für SHAP (Sample für Speed)
sample_idx = np.random.choice(len(X_train_cleaned), 20_000, replace=False)
rf = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_cleaned.iloc[sample_idx], y_train[sample_idx])

feature_importance_analyse(
    model=rf,
    X_test=X_test_cleaned,
    output_dir=str(root_dir / '05_plots'),
    filename='shap_plot.png'
)

Beide Plots wurden in ..\05_plots gespeichert.


## 4.5 Reduziertes Feature-Set

Auf Basis der SHAP-Analyse werden folgende Features entfernt:
- `heating_amount`: Kaum Informationsgehalt durch hohe Missings
- `building_type_*`: SHAP ≈ 0
- `solar_thermal_potential`: Keine Varianz
- `is_weekend`: Minimaler Effekt
- `renovation_score`: Kein Einfluss
- `is_heating_season`: Redundant zu `temperature_avg_daily`

In [6]:
X_train_red = X_train[feature_cols_cleaned]
X_test_red  = X_test[feature_cols_cleaned]

X_train_final, X_test_final, final_features = correlated_features_drop(
    X_train_red, X_test_red, feautre_cols=feature_cols_cleaned
)
print(f'Finales Feature-Set ({len(final_features)}): {final_features}')

Entfernte redundante Features: []
Verbleibende Features (10): ['temperature_avg_daily', 'kwh_returned_total', 'temp_delta_1d', 'kwh_received_heatpump', 'price_lag_30d', 'price_relative_to_month', 'swissix_base', 'weekday', 'month_sin', 'month_cos']
Finales Feature-Set (10): ['temperature_avg_daily', 'kwh_returned_total', 'temp_delta_1d', 'kwh_received_heatpump', 'price_lag_30d', 'price_relative_to_month', 'swissix_base', 'weekday', 'month_sin', 'month_cos']


## 4.6 Modellvergleich

In [9]:
comparison = compare_models(X_train_final, X_test_final, y_train, y_test)
print(comparison)

--- Benchmark gestartet (Sample-Größe: 5000) ---
  -> Teste Ridge (Linear)... Fertig (0.01s)
  -> Teste XGBoost (Fast)... Fertig (0.24s)
  -> Teste LightGBM... Fertig (0.5s)
  -> Teste RandomForest (Limited)... Fertig (0.35s)
                    Model  MAE_kWh  R2_Score  Duration_sec
3  RandomForest (Limited)   12.148     0.308          0.35
2                LightGBM   12.296     0.293          0.50
0          Ridge (Linear)   12.693     0.285          0.01
1          XGBoost (Fast)   13.185     0.189          0.24


Der Benchmark zeigt, dass RandomForest mit einem MAE von 12,15 kWh und einem 
𝑅
2
R
2
 von 0,308 die beste Prognosegüte erzielt. LightGBM erreicht mit einem MAE von 12,30 kWh und einem 
𝑅
2
R
2
 von 0,293 ein nahezu gleichwertiges Ergebnis, während Ridge und XGBoost deutlich schlechter abschneiden. Aufgrund der vergleichbaren Performance sowie der besseren Skalierbarkeit und Interpretierbarkeit wurde LightGBM als finales Modell ausgewählt.

## 4.7 LightGBM Tuning – Standard

In [10]:
lgbm_model_std, study_std = tune_lightgbm(
    X_train=X_train_final, y_train=y_train,
    X_test=X_test_final, y_test=y_test,
    final_features=final_features,
    output_dir=str(root_dir / '05_plots'),
    n_trials=50, log_transform=False
)

🔍 Starte Optuna Tuning...


Best trial: 35. Best value: 11.5934: 100%|██████████| 50/50 [19:39<00:00, 23.58s/it]


✅ Bestes MAE (CV): 11.593 kWh
✅ Beste Parameter: {'n_estimators': 778, 'learning_rate': 0.17574440002741754, 'num_leaves': 107, 'max_depth': 3, 'min_child_samples': 83, 'subsample': 0.6594946156611715, 'colsample_bytree': 0.672160604077144, 'reg_alpha': 8.141674351170213e-07, 'reg_lambda': 1.5638657688968143e-07}
📊 Lernkurven gespeichert: ..\05_plots\lgbm_learning_curve_std.png

📈 Test-Ergebnisse:
   MAE:  11.957 kWh
   R²:   0.341
💾 Modell gespeichert: ..\05_plots\lgbm_final_model_std.pkl


## 4.8  LightGBM Tuning – Log-Transformation

In [11]:
lgbm_model_log, study_log = tune_lightgbm(
    X_train=X_train_final, y_train=y_train,
    X_test=X_test_final, y_test=y_test,
    final_features=final_features,
    output_dir=str(root_dir / '05_plots'),
    n_trials=50, log_transform=True
)

🔍 Starte Optuna Tuning...


Best trial: 0. Best value: 11.2852:   2%|▏         | 1/50 [00:15<12:43, 15.58s/it]


[W 2026-03-30 17:44:26,203] Trial 1 failed with parameters: {'n_estimators': 252, 'learning_rate': 0.0493272479826721, 'num_leaves': 38, 'max_depth': 11, 'min_child_samples': 51, 'subsample': 0.5489225471909236, 'colsample_bytree': 0.7386563803589157, 'reg_alpha': 0.00019749272958230715, 'reg_lambda': 0.11360556371855467} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\maxkr\.pyenv\pyenv-win\versions\3.11.6\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "c:\Users\maxkr\HEAPO-ConsumptionPredict\04_notebooks\..\03_src\utilis.py", line 261, in objective
    model.fit(
  File "c:\Users\maxkr\.pyenv\pyenv-win\versions\3.11.6\Lib\site-packages\lightgbm\sklearn.py", line 1398, in fit
    super().fit(
  File "c:\Users\maxkr\.pyenv\pyenv-win\versions\3.11.6\Lib\site-packages\lightgbm\sklearn.py", line 1049, in fit
    self._Booster = train(
 

KeyboardInterrupt: 

## 4.9 Modellvergleich Standard vs. Log

In [ ]:
y_pred_std = lgbm_model_std.predict(X_test_final)
y_pred_log = np.expm1(lgbm_model_log.predict(X_test_final))

mae_std = mean_absolute_error(y_test, y_pred_std)
mae_log = mean_absolute_error(y_test, y_pred_log)

print(f'Standard – MAE: {mae_std:.3f} kWh | R²: {r2_score(y_test, y_pred_std):.3f}')
print(f'Log      – MAE: {mae_log:.3f} kWh | R²: {r2_score(y_test, y_pred_log):.3f}')

if mae_log < mae_std:
    print('✅ Log-Transformation ist besser')
    lgbm_model = lgbm_model_log
    y_pred = y_pred_log
else:
    print('✅ Standard ist besser')
    lgbm_model = lgbm_model_std
    y_pred = y_pred_std